<a href="https://colab.research.google.com/github/scudilio/AI---Deep-Learning/blob/main/Aula%204/MBA_DL_AULA4_04_fine_tuning_lora_bert.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Aula 4 | Case 3: fine-tuning com LoRA, o BERT vira especialista em reviews

**MBA FIAP | Deep Learning e Redes Neurais Modernas | Profa. Juliana Scudilio**

**Problema de negócio.** O classificador de sentimento vai rodar em milhões de mensagens por mês, precisa ser rápido, estável e o melhor possível. O prompt (case 1) é lento e o RAG (case 2) não serve: o problema não é conhecimento, é **comportamento**. Queremos um modelo que faça uma única coisa muito bem.

**A ideia do método.** Transfer learning, como na aula 1 com imagens: um modelo pré-treinado (BERTimbau, que leu bilhões de palavras em português) é **ajustado** com poucos exemplos do nosso problema. No notebook 00 desta aula congelamos o BERT e treinamos só uma regressão logística em cima (92,9% com 1.500 reviews). Hoje vamos mexer nos pesos do próprio BERT, mas de um jeito barato: **LoRA** (*Low-Rank Adaptation*) congela as matrizes originais e treina matrizes pequenas ao lado delas. Treinamos 0,3% dos parâmetros e o resultado se guarda num arquivo de 1 MB.

**O que vamos medir.** Acurácia nos mesmos 1.000 reviews de teste do notebook 00 e do case 1, contra todas as abordagens vistas até aqui; parâmetros treinados; tempo; tamanho do que precisa ser guardado.

**Roteiro (45 min).** 0. Preparar → 1. Dados → 2. O BERT inteiro: quantos parâmetros → 3. LoRA: quantos parâmetros → 4. Texto em tokens → 5. O laço de treino (o backprop da aula 1) → 6. Medir no teste → 7. Comparar tudo → 8. Frases da turma → 9. Guardar o adaptador.

Em CPU, o treino leva uns 6 minutos (2 épocas). Com GPU no Colab, menos de 1 minuto.

## 0. Preparar o ambiente

`peft` (*Parameter-Efficient Fine-Tuning*) é a biblioteca da Hugging Face que implementa LoRA e parentes. Ela recebe um modelo normal e devolve o mesmo modelo com as matrizes pequenas acopladas.

In [1]:
%pip uninstall -y -q torchao                          # o Colab traz um torchao antigo (0.10) que o peft rejeita; sem ele, o peft funciona
%pip install -q "transformers>=4.56" peft accelerate

import os                                  # achar o arquivo
import time                                # cronometrar
import numpy as np                         # contas
import pandas as pd                        # tabelas
import torch                               # o motor: tensores, gradientes, otimizador
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from peft import LoraConfig, get_peft_model

torch.manual_seed(0)                       # mesmos sorteios (inicialização do LoRA, ordem dos lotes)
np.random.seed(0)
print("GPU disponível?", torch.cuda.is_available())

GPU disponível? False


## 1. Dados: a mesma amostra do notebook 00

`df.sample(2500, random_state=0)`: os mesmos 2.500 reviews do notebook 00 (BERT congelado) e do case 1. Os 1.500 primeiros treinam; os 1.000 últimos testam. Assim, cada número da tabela final foi medido **nos mesmos reviews**.

In [2]:
caminho = None
for c in ["reviews_b2w_subset.csv", "../data/reviews_b2w_subset.csv", "data/reviews_b2w_subset.csv"]:
    if os.path.exists(c):
        caminho = c
if caminho is None:
    from google.colab import files
    print("Envie o arquivo reviews_b2w_subset.csv:")
    caminho = list(files.upload())[0]

df = pd.read_csv(caminho)
df["review_text"] = df["review_text"].astype(str)
df["y"] = (df["sentimento"] == "positivo").astype(int)     # 1 = positivo, 0 = negativo

amostra = df.sample(2500, random_state=0)
tr = amostra.iloc[:1500]                   # treino: 1.500 reviews
te = amostra.iloc[1500:]                   # teste: 1.000 reviews
print("treino", len(tr), "| teste", len(te), "| positivos no treino:", f"{tr['y'].mean():.0%}")

treino 1500 | teste 1000 | positivos no treino: 50%


## 2. O BERT inteiro: 109 milhões de parâmetros

`AutoModelForSequenceClassification` carrega o BERTimbau e acrescenta em cima uma **cabeça de classificação**: uma camada densa nova que recebe o vetor do token `[CLS]` (768 números) e devolve 2 números, um por classe. Essa cabeça começa com pesos aleatórios: é por isso que o aviso diz que `classifier.weight` foi "newly initialized". Tudo o resto vem do pré-treino.

Fine-tuning completo = ajustar todos os 109 milhões. Funciona, mas: precisa de GPU com memória, cada versão do modelo pesa 440 MB, e com 1.500 exemplos o risco de **esquecimento catastrófico** (o modelo perde o que sabia de português para decorar os exemplos) é real.

In [3]:
NOME = "neuralmind/bert-base-portuguese-cased"                   # BERTimbau, o mesmo do notebook 00

tokenizador = AutoTokenizer.from_pretrained(NOME)
modelo = AutoModelForSequenceClassification.from_pretrained(NOME, num_labels=2)   # BERT + cabeça de 2 classes

total = sum(p.numel() for p in modelo.parameters())
print(f"Parâmetros do BERT + cabeça: {total:,}")
print(f"Tamanho em disco, se salvarmos tudo em float32: {total * 4 / 1e6:.0f} MB")

config.json:   0%|          | 0.00/647 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/43.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/210k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  438MB            

... (2 outputs left out)

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: neuralmind/bert-base-portuguese-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from th

Parâmetros do BERT + cabeça: 108,924,674
Tamanho em disco, se salvarmos tudo em float32: 436 MB


## 3. LoRA: treinar 0,3% e congelar o resto

Cada camada de atenção do BERT tem as matrizes Q, K e V que vimos no notebook 00 (768 × 768 = 590 mil números cada). LoRA congela a matriz original $W$ e coloca ao lado duas matrizes finas, $A$ (768 × r) e $B$ (r × 768), com $r$ pequeno. O que o modelo usa é $W + A \cdot B$. Só $A$ e $B$ aprendem. Com $r = 8$: 768 × 8 + 8 × 768 = 12.288 números por matriz, em vez de 590 mil. 48 vezes menos.

Os parâmetros do `LoraConfig`:

- `r=8`: a "espessura" das matrizes finas. 4 a 16 é o usual; maior = mais capacidade e mais parâmetros;
- `lora_alpha=16`: um fator de escala do que $A \cdot B$ acrescenta (regra prática: 2 × r);
- `target_modules=["query", "value"]`: em quais matrizes acoplar. Q e V são o padrão do artigo original;
- `lora_dropout=0.1`: desliga 10% ao acaso no treino, contra decorar;
- `task_type="SEQ_CLS"`: avisa que a cabeça de classificação também deve ser treinada (ela é nova; tem que aprender).

In [4]:
config = LoraConfig(
    r=8,                                   # posto (espessura) das matrizes A e B
    lora_alpha=16,                         # escala do ajuste
    lora_dropout=0.1,                      # regularização
    target_modules=["query", "value"],     # onde acoplar: as matrizes Q e V de cada camada de atenção
    task_type="SEQ_CLS",                   # classificação de sequência: treina a cabeça também
)
modelo = get_peft_model(modelo, config)    # o mesmo modelo, agora com as matrizes LoRA acopladas e o resto congelado

treinaveis = sum(p.numel() for p in modelo.parameters() if p.requires_grad)
print(f"Parâmetros treináveis: {treinaveis:,} de {total:,} ({treinaveis / total:.2%})")
print(f"Tamanho do adaptador em disco: ~{treinaveis * 4 / 1e6:.1f} MB (contra {total * 4 / 1e6:.0f} MB do modelo inteiro)")

Parâmetros treináveis: 296,450 de 108,924,674 (0.27%)
Tamanho do adaptador em disco: ~1.2 MB (contra 436 MB do modelo inteiro)


### Onde as matrizes foram acopladas

Olhe uma camada de atenção: `query` e `value` agora são `lora.Linear`, com `base_layer` (a original, congelada) e `lora_A` / `lora_B` (as novas, treináveis). `key` ficou como era.

In [5]:
print(modelo.base_model.model.bert.encoder.layer[0].attention.self)

BertSelfAttention(
  (query): lora.Linear(
    (base_layer): Linear(in_features=768, out_features=768, bias=True)
    (lora_dropout): ModuleDict(
      (default): Dropout(p=0.1, inplace=False)
    )
    (lora_A): ModuleDict(
      (default): Linear(in_features=768, out_features=8, bias=False)
    )
    (lora_B): ModuleDict(
      (default): Linear(in_features=8, out_features=768, bias=False)
    )
    (lora_embedding_A): ParameterDict()
    (lora_embedding_B): ParameterDict()
    (lora_magnitude_vector): ModuleDict()
  )
  (key): Linear(in_features=768, out_features=768, bias=True)
  (value): lora.Linear(
    (base_layer): Linear(in_features=768, out_features=768, bias=True)
    (lora_dropout): ModuleDict(
      (default): Dropout(p=0.1, inplace=False)
    )
    (lora_A): ModuleDict(
      (default): Linear(in_features=768, out_features=8, bias=False)
    )
    (lora_B): ModuleDict(
      (default): Linear(in_features=8, out_features=768, bias=False)
    )
    (lora_embedding_A): Param

## 4. Texto em tokens

O mesmo tokenizador do notebook 00: quebra em subpalavras (`##`), põe `[CLS]` no início e `[SEP]` no fim. `max_length=64`: reviews com mais de 64 tokens são cortados (a maioria cabe; e o custo de atenção cresce com o quadrado do comprimento). `padding=True` preenche os mais curtos; `attention_mask` diz ao modelo o que é preenchimento.

In [6]:
MAX_TOKENS = 64

enc_tr = tokenizador(list(tr["review_text"]), truncation=True, max_length=MAX_TOKENS, padding=True, return_tensors="pt")
enc_te = tokenizador(list(te["review_text"]), truncation=True, max_length=MAX_TOKENS, padding=True, return_tensors="pt")
y_tr = torch.tensor(tr["y"].values)        # rótulos como tensor
y_te = te["y"].values

print("Treino:", enc_tr["input_ids"].shape, "| teste:", enc_te["input_ids"].shape)
print("Exemplo:", tokenizador.convert_ids_to_tokens(enc_tr["input_ids"][0])[:16])

Treino: torch.Size([1500, 64]) | teste: torch.Size([1000, 64])
Exemplo: ['[CLS]', 'A', 'cor', '##tina', 'não', 'corre', '##n', '##pon', '##de', 'a', 'fidelidade', 'da', 'imagem', '!', 'Que', 'mostra']


## 5. O laço de treino: o backprop da aula 1, escrito à mão

Na aula 1 o `fit` do Keras escondia isto. Aqui está o laço explícito, porque é curto e porque é literalmente o que a aula 1 descreveu:

1. pega um **lote** de 16 reviews;
2. **forward**: o modelo calcula as previsões e a perda (`out.loss`, a entropia cruzada entre previsão e rótulo);
3. **backward**: `loss.backward()` calcula, para cada parâmetro treinável, o quanto ele contribuiu para o erro (o gradiente, a "atribuição de culpa");
4. **passo**: `otimizador.step()` ajusta os parâmetros na direção que reduz o erro, com o tamanho de passo `lr`;
5. zera os gradientes e vai para o próximo lote.

Uma **época** é uma passagem pelos 1.500 reviews (94 lotes). Fazemos 2. O otimizador é o AdamW (Adam com decaimento de pesos), o padrão para Transformers. `lr=3e-4` é alto para fine-tuning completo e adequado para LoRA, porque só as matrizes pequenas se movem.

Repare que só os parâmetros com `requires_grad` vão para o otimizador: os 109 milhões congelados nem entram.

In [7]:
LOTE = 16
EPOCAS = 2
otimizador = torch.optim.AdamW([p for p in modelo.parameters() if p.requires_grad], lr=3e-4)   # só os treináveis

modelo.train()                                                       # modo treino (dropout ligado)
t0 = time.time()
for epoca in range(EPOCAS):
    ordem = torch.randperm(len(y_tr))                                # embaralha os 1.500 a cada época
    perdas = []
    for i in range(0, len(y_tr), LOTE):
        j = ordem[i:i + LOTE]                                        # índices do lote
        out = modelo(input_ids=enc_tr["input_ids"][j],               # forward: previsões e perda do lote
                     attention_mask=enc_tr["attention_mask"][j],
                     labels=y_tr[j])
        out.loss.backward()                                          # backward: gradiente de cada parâmetro treinável
        otimizador.step()                                            # passo: ajusta os parâmetros
        otimizador.zero_grad()                                       # limpa os gradientes para o próximo lote
        perdas.append(out.loss.item())
        if (i // LOTE) % 20 == 0:                                    # a cada 20 lotes, mostra a perda
            print(f"época {epoca + 1} | lote {i // LOTE + 1:3d} | perda {out.loss.item():.3f} | {time.time() - t0:.0f} s")
    print(f"=== época {epoca + 1} terminada | perda média {np.mean(perdas):.3f} | {time.time() - t0:.0f} s\n")
tempo_treino = time.time() - t0

época 1 | lote   1 | perda 0.675 | 24 s
época 1 | lote  21 | perda 0.591 | 226 s
época 1 | lote  41 | perda 0.488 | 382 s
época 1 | lote  61 | perda 0.093 | 537 s
época 1 | lote  81 | perda 0.094 | 694 s
=== época 1 terminada | perda média 0.423 | 794 s

época 2 | lote   1 | perda 0.031 | 803 s
época 2 | lote  21 | perda 0.053 | 961 s
época 2 | lote  41 | perda 0.328 | 1117 s
época 2 | lote  61 | perda 0.262 | 1270 s
época 2 | lote  81 | perda 0.044 | 1422 s
=== época 2 terminada | perda média 0.167 | 1520 s



⏱ **Checkpoint 1:** a perda média caiu da primeira para a segunda época? Quanto tempo levou?

## 6. Medir no teste

`modelo.eval()` desliga o dropout. `torch.no_grad()` desliga o cálculo de gradientes (só inferência, mais rápido). A saída `logits` tem 2 números por review; `argmax` escolhe a classe com o maior.

In [8]:
modelo.eval()
palpites = []
t0 = time.time()
with torch.no_grad():
    for i in range(0, len(y_te), 64):                                # lotes de 64 na inferência
        out = modelo(input_ids=enc_te["input_ids"][i:i + 64], attention_mask=enc_te["attention_mask"][i:i + 64])
        palpites.append(out.logits.argmax(dim=1))                    # a classe com maior pontuação
palpites = torch.cat(palpites).numpy()
tempo_inf = time.time() - t0

acc_lora = (palpites == y_te).mean()
print(f"Teste: acurácia {acc_lora:.1%} | {tempo_inf / len(y_te) * 1000:.1f} ms por review")

Teste: acurácia 94.3% | 203.7 ms por review


## 7. Comparar tudo: o que cada degrau comprou

Todas as linhas abaixo foram medidas nos mesmos reviews de teste (1.000 da amostra; 200 deles no case 1). As duas primeiras vêm do notebook 00, as do LLM do case 1. Preencha as do case 1 com os seus números se forem diferentes.

In [9]:
comp = pd.DataFrame([
    ["TF-IDF + regressão logística (notebook 00)",  "1.500 rotulados",       "89,1%", "0",        "< 1 ms"],
    ["BERT congelado + regressão logística (notebook 00)", "1.500 rotulados",     "92,9%", "769 (a regressão)", "~ 60 ms"],
    ["LLM 0,5B zero-shot (case 1, 200 reviews)",    "0 rotulados",           "92,5%", "0",        "~ 550 ms"],
    ["LLM 0,5B few-shot, 4 exemplos (case 1)",      "4 exemplos no prompt",  "95,0%", "0",        "~ 1.000 ms"],
    ["BERT + LoRA (este notebook)",                 "1.500 rotulados",       f"{acc_lora:.1%}".replace(".", ","), f"{treinaveis:,}".replace(",", "."), f"~ {tempo_inf / len(y_te) * 1000:.0f} ms"],
], columns=["abordagem", "dados", "acurácia (teste)", "parâmetros treinados", "tempo por review (CPU)"])
comp

,abordagem,dados,acurácia (teste),parâmetros treinados,tempo por review (CPU)
0,TF-IDF + regressão logística (notebook 00),1.500 rotulados,"89,1%",0,< 1 ms
1,BERT congelado + regressão logística (notebook...,1.500 rotulados,"92,9%",769 (a regressão),~ 60 ms
2,"LLM 0,5B zero-shot (case 1, 200 reviews)",0 rotulados,"92,5%",0,~ 550 ms
3,"LLM 0,5B few-shot, 4 exemplos (case 1)",4 exemplos no prompt,"95,0%",0,~ 1.000 ms
4,BERT + LoRA (este notebook),1.500 rotulados,"94,3%",296.450,~ 204 ms


**Leitura de negócio.** Com os mesmos 1.500 reviews, ajustar 0,3% do BERT rendeu mais do que congelá-lo (notebook 00) e ficou a menos de um ponto do LLM few-shot (medido em só 200 reviews, com margem de erro maior), sendo cerca de 20 vezes mais rápido por review. O preço: precisa de dados rotulados, de um treino (minutos em GPU) e de reavaliação a cada mudança de dado. O prompt começa amanhã; o fine-tuning ganha quando o volume e a estabilidade justificam.

**Esquecimento catastrófico, na prática.** Como só as matrizes pequenas mudaram, o BERT original está intacto dentro do modelo. Se o ajuste piorar algo, basta descartar o adaptador. Com fine-tuning completo, não há volta sem a cópia original.

## 8. Frases da turma

As frases que enganaram o TF-IDF e a LSTM na aula 3 e o BERT congelado no notebook 00. `softmax` transforma os 2 logits em probabilidades que somam 1; mostramos a de "positivo".

In [10]:
frases = [
    "Produto excelente, chegou antes do prazo.",
    "Não gostei, veio com defeito e o SAC não resolve.",
    "Chegou rápido, mas parou de funcionar em uma semana.",
    "Não é ruim.",
    "Esperava mais pelo preço.",
    "Produto bom, entrega péssima.",
]
enc = tokenizador(frases, truncation=True, max_length=MAX_TOKENS, padding=True, return_tensors="pt")
with torch.no_grad():
    prob = modelo(**enc).logits.softmax(dim=1)[:, 1]                # probabilidade de positivo
for frase, p in zip(frases, prob):
    print(f"{p:5.0%} positivo | {frase}")

  99% positivo | Produto excelente, chegou antes do prazo.
   2% positivo | Não gostei, veio com defeito e o SAC não resolve.
   6% positivo | Chegou rápido, mas parou de funcionar em uma semana.
  41% positivo | Não é ruim.
  59% positivo | Esperava mais pelo preço.
   8% positivo | Produto bom, entrega péssima.


## 9. Guardar o adaptador

`save_pretrained` de um modelo PEFT salva **só as matrizes LoRA e a cabeça**, não o BERT. Para usar em produção: carrega-se o BERTimbau original do Hub e aplica-se o adaptador por cima. Dez tarefas diferentes = dez adaptadores de 1 MB sobre o mesmo modelo base de 440 MB.

In [11]:
modelo.save_pretrained("adaptador_lora_sentimento")                 # pasta com o adaptador
tokenizador.save_pretrained("adaptador_lora_sentimento")

tamanho = sum(os.path.getsize(os.path.join("adaptador_lora_sentimento", f)) for f in os.listdir("adaptador_lora_sentimento"))
print(f"Arquivos salvos: {os.listdir('adaptador_lora_sentimento')}")
print(f"Tamanho total: {tamanho / 1e6:.1f} MB (o BERT inteiro teria {total * 4 / 1e6:.0f} MB)")

import json
resultado = {"modelo_base": NOME, "lora_r": 8, "n_treino": len(tr), "epocas": EPOCAS, "parametros_treinaveis": int(treinaveis),
             "acc_teste": round(float(acc_lora), 4), "tempo_treino_s": round(tempo_treino), "ms_por_review": round(tempo_inf / len(y_te) * 1000, 1)}
json.dump(resultado, open("resultado_case3_lora.json", "w"), indent=2)
print(resultado)

Arquivos salvos: ['adapter_config.json', 'tokenizer.json', 'adapter_model.safetensors', 'README.md', 'tokenizer_config.json']
Tamanho total: 1.9 MB (o BERT inteiro teria 436 MB)
{'modelo_base': 'neuralmind/bert-base-portuguese-cased', 'lora_r': 8, 'n_treino': 1500, 'epocas': 2, 'parametros_treinaveis': 296450, 'acc_teste': 0.943, 'tempo_treino_s': 1520, 'ms_por_review': 203.7}


## Para discutir

1. O LoRA rodou com 1.500 reviews. Quantos exemplos rotulados a sua empresa tem para o problema que quer resolver? Quem rotula?
2. Dez áreas, dez tarefas, um modelo base: o que isso muda em infraestrutura e governança?
3. O few-shot do case 1 empatou sem treinar. Em que cenário você escolheria cada um, mesmo com acurácia igual?

**Próximo case:** o mesmo raciocínio em imagens, com um modelo que lê imagem e texto no mesmo espaço (CLIP).